## Installation

In [ ]:
!pip install flash_attn==2.7.4.post1
!pip install torch==2.5.1
!pip install transformers==4.49.0
!pip install accelerate==1.3.0
!pip install sentence-transformers

!pip install chromadb

In [ ]:
import pandas as pd

import chromadb
from chromadb.utils.embedding_functions import EmbeddingFunction
from sentence_transformers import SentenceTransformer

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)

## Read Dataset

In [ ]:
df_corpus = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_que = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [ ]:
print("df_corpus:")
print(df_corpus.head())

print("\ndf_que:")
print(df_que.head())

In [ ]:
df_corpus.describe()

## Process the Documents 

In [ ]:
df_corpus.dropna(inplace=True)

In [ ]:
documents=[]
for index,row in df_corpus.iterrows():
    documents.append({
        "id":str(index),
        "text":row["passage"]
    })
documents[:3]

## Model Embedding

In [ ]:
embedding_model = SentenceTransformer(
    "jinaai/jina-embeddings-v2-small-en",
    trust_remote_code=True,
)

class CustomEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model):
        self.model = model  
        self.model.max_seq_length = 1024
    
    def __call__(self, texts):
        return self.model.encode(texts,batch_size=1) 

custom_embedder = CustomEmbeddingFunction(embedding_model)

## Vector Database - ChromaDB 

In [ ]:
chroma_client = chromadb.PersistentClient(path="/kaggle/working/chroma_db")

collection = chroma_client.get_or_create_collection(name="my_documents", embedding_function=custom_embedder)

In [ ]:
def batch_add(collection, documents, batch_size=5000):
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]

        collection.add(
            ids=[doc["id"] for doc in batch],
            documents=[doc["text"] for doc in batch]
        )

        print(f"Added batch {i} → {i + len(batch)}")


In [ ]:
batch_add(collection, documents, batch_size=5000)

## Document retrieving relevant to the query.

In [ ]:
retrieved_data=collection.query(
    query_texts=["Is the protein Papilin secreted?"],
    n_results=10
)

In [ ]:
retrieved_data

## Loading language model 

In [ ]:

model_path = "microsoft/Phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="balanced",
    torch_dtype="auto",
    trust_remote_code=True,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

## Generating text 

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "hi,how are you"},
]
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
}
 
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

## Building Rag System

In [ ]:
class RAG():
    def __init__(self,retriever,llm):
        self.retriever=retriever
        self.llm=llm
        self.system_prompt="""You are a helpful AI assistant."""
        self.user_prompt="""You are an assistant for question-answering tasks. Use only the following pieces of retrieved context to answer the question.
        If the context doesnt contain the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
        Question: {} 
        Context: {} 
        Answer:"""
        self.generation_args= {
                            "max_new_tokens": 500,
                            "return_full_text": False,
                        }
    def answer_query(self,query,top_k=5):
        retrieved_documents=self.retriever.query(
                                            query_texts=[query],
                                            n_results=top_k
                                            )["documents"][0]
        retrieved_information="\n".join(retrieved_documents)
        messages = [
        {"role": "system", "content": self.system_prompt},
        {"role": "user", "content": self.user_prompt.format(query,retrieved_information)}]
        output=self.llm(messages, **self.generation_args)
        return output[0]['generated_text']
        

In [ ]:
query=df_que.iloc[2]["question"]
print("query:",query)
print("answer:",df_que.iloc[2]["answer"])


In [ ]:
rag=RAG(collection,pipe)
rag.answer_query("Is the protein Papilin secreted?",10)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Is the protein Papilin secreted?"},
]
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
}
 
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])